# Probe iPhone: pipeline label MuQ saja (tanpa MusicFM)

Buka dari root **SongMap**; memakai Python `.venv-songformer311` di sibling Section-Annotations. Butuh Xcode dan iPhone fisik.

Membuat app uji terpisah **MuQ Only** (`com.danendra.MuQOnlyPipelineProbe`) berisi tepat lima model: log-mel dan konvolusi MuQ dari app, bank MuQ canonical (sama persis dengan yang divalidasi), head palette6 input 2048, dan konteks FP32 input 2048. MusicFM tidak dibundel. Runner sama dengan probe canonical dua-encoder (jadwal serial, owned FP32, frontend & partial di CPU, CPU+GPU hanya untuk window penuh), sehingga waktu bisa dibandingkan langsung.

Sebelum membuat proyek, di Mac: tes native Swift (owned tensor, fusi dua stream, pooling 2048), typecheck SDK iPhone, dan referensi Mac Core ML untuk dua lagu (Alleluia dan Raise a Hallelujah) yang harus mereproduksi validasi 32 lagu (label identik, probabilitas akhir ≤ 1e-4). Gerbang di iPhone tidak berubah: label sama dengan Mac, probabilitas base dan akhir ≤ 1e-4.

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, subprocess
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/prepare_muq_pipeline_iphone.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
QUALITY=sorted(p for p in (ROOT/'outputs.local').glob('muq-encoder-quality-*/validation-all.json') if json.loads(p.read_text()).get('passed'))[-1]
EXPORT=sorted(p for p in (ROOT/'outputs.local').glob('muq-coreml-export-*') if (p/'export_report.json').is_file())[-1]
CANONICAL=next(p for p in (ROOT/'outputs.local').glob('encoder-canonical-bank-*/quality/validation-all.json')
               if sha(p)=='020f8d9a2bb5c29318f591a7c58d30b8f4cdc39eba47cf82c09820b538ac96fa')
wanted=json.loads((CANONICAL.parent/'selection.json').read_text())['reports']['muq']
MUQ_REPORT=next(p for p in CANONICAL.parent.parent.rglob('*.json') if p.stat().st_size<5_000_000 and sha(p)==wanted)
OUTPUT=ROOT/'outputs.local'/('muq-pipeline-iphone-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
print('Validasi MuQ:',QUALITY); print('Ekspor:',EXPORT); print('Bank MuQ:',MUQ_REPORT); print('Proyek baru:',OUTPUT)

In [ ]:
log=Path(str(OUTPUT)+'.prepare.log')
command=[str(RUN_PYTHON),'-u','scripts/prepare_muq_pipeline_iphone.py','--songstruct-root',str(SONGSTRUCT_ROOT),
    '--songmap-root',str(ROOT),'--quality-report',str(QUALITY),'--export-dir',str(EXPORT),
    '--muq-report',str(MUQ_REPORT),'--output-dir',str(OUTPUT)]
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status: raise RuntimeError(f'Persiapan gagal ({status}); lihat {log}. Kirim mac-preflight.json bila ada.')
proof=json.loads((OUTPUT/'preparation.json').read_text())
assert proof['native_swift_checks_passed'] and proof['native_typecheck_passed'] and proof['mac_reference_passed']
assert proof['musicfm_bundled'] is False
for check in proof['checks']: print(check['track_id'],'Mac reference:',check['passed'],'max final error',check['max_final_probability_error'])
print('Model MiB:',{k:round(v/2**20,1) for k,v in proof['compiled_model_bytes'].items()},'| total',round(proof['temporary_model_bytes']/2**20,1))
subprocess.run(['open',str(OUTPUT/'Benchmark.xcodeproj')],check=True)

**Urutan di iPhone** (sama seperti probe dua-encoder, agar waktu bisa dibandingkan):

1. Pilih *Raise a Hallelujah*, mode **CPU only**, *Run selected track*. Pastikan `label_parity_passed: true`. Bagikan JSON.
2. Tutup app sepenuhnya, buka lagi. Mode **CPU only**, *Run 3 passes over all tracks* (6 run). Bagikan batch JSON.
3. Tutup app, buka lagi. Mode **CPU + GPU candidate**, *Run 3 passes over all tracks*. Bagikan batch JSON.

Pembanding dua-encoder (CPU, batch sebelumnya): Alleluia 46,1 / 45,5 / 51,2 detik; Raise a Hallelujah 113,8 / 124,8 / 147,5 detik. Bila app berhenti, buka lagi dan bagikan *last recorded stage*.